In [385]:
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib

Using matplotlib backend: Qt5Agg


In [74]:
def chk_tol(val, target, tol):
#   Returns True if val is within target +/- tol, otherwise false.
    return 1 if target - tol <= val <= target + tol else 0

In [414]:
inf = r'C:\Users\nicfran\PycharmProjects\b0t1y\Dumps\Mar 25 2022 130631_botley_ir_2_decoded_History.txt'
rules = {'header': 4150, 'mark': 642, 'space0': 1284, 'space1': 642, 'bits': 16, 'delta': 200, 'quiet': 5000, 'carrier': 38000}

# Load the saved session
lines = []
with open(inf, 'r') as fh:
    lines = fh.readlines()

In [76]:
# Parse the saved session
commands = {} 
for command in [line.split(',') for line in lines[1:]]:
    button = command[3]
    pulse_data = [np.array(sample.split(';')[1:], dtype=int) for sample in command[8].split('|')][:-1]
    if button in commands.keys():
        commands[button].append(pulse_data)
    else:
        commands[button] = [pulse_data]

In [394]:
commands

{'send_dall_30x': [[array([   0, 1000,    0]),
   array([1000, 4132,    1]),
   array([5132,  716,    0]),
   array([5848,  576,    1]),
   array([6424,  692,    0]),
   array([7116,  576,    1]),
   array([7692,  716,    0]),
   array([8408,  572,    1]),
   array([8980,  720,    0]),
   array([9700,  552,    1]),
   array([10252,   740,     0]),
   array([10992,   576,     1]),
   array([11568,  1340,     0]),
   array([12908,   548,     1]),
   array([13456,   744,     0]),
   array([14200,   576,     1]),
   array([14776,  1336,     0]),
   array([16112,   576,     1]),
   array([16688,   720,     0]),
   array([17408,   552,     1]),
   array([17960,   740,     0]),
   array([18700,   576,     1]),
   array([19276,   716,     0]),
   array([19992,   576,     1]),
   array([20568,  1340,     0]),
   array([21908,   576,     1]),
   array([22484,   716,     0]),
   array([23200,   576,     1]),
   array([23776,   716,     0]),
   array([24492,   576,     1]),
   array([25068,  1340,

In [77]:
# Print the number of captures per transmission type
[f'{len(commands[key])}, {key}' for key in commands.keys()]

['5, send_F_B_F_B_confirm_pair',
 '1, send_L4_F_R4_L9_B_R9_after_pair',
 '1, clear_after_pair',
 '2, pairing_init',
 '1, pairing_timeout',
 '7, Clear',
 '1, botley_says',
 '2, sound_L-O',
 '2, sound_H-L',
 '2, sound_O-H',
 '1, send_L4_F_R4_L9_B_R9_OD_ON_L4_F_R4_L9_B_R9_OD-OFF_L4_F_R4_L9_B_R9',
 '2, OD_Disable',
 '2, OD_Enable',
 '2, clear_L4_F_R4_L9_B_R9',
 '3, send_L4_F_R4_L9_B_R9',
 '9, light',
 '1, send_FFFF',
 '1, send_F',
 '1, send_empty',
 '2, loop-clear',
 '3, loop']

In [438]:
inf = r'C:\Users\nicfran\PycharmProjects\b0t1y\Dumps\Mar 25 2022 130631_botley_ir_2_decoded_History.txt'
rules = {'header': 4150, 'mark': 642, 'space0': 1284, 'space1': 642, 'bits': 16, 'delta': 200, 'quiet': 25000, 'carrier': 38000}

# Load the saved session
lines = []
with open(inf, 'r') as fh:
    lines = fh.readlines()

# Parse the saved session
commands = {} 
for command in [line.split(',') for line in lines[1:]]:
    button = command[3]
    pulse_data = [np.array(sample.split(';')[1:], dtype=int) for sample in command[8].split('|')][:-1]
    if button in commands.keys():
        commands[button].append(pulse_data)
    else:
        commands[button] = [pulse_data]

# Decode the contents of each transmission sample
decoded = {}
for command, history in commands.items():
    samples = []
    for series in history:
        bursts = []
        burst = []
        header = False
        bits = []
        
        for sequence in series:
            if not sequence[2]:                   # If a space
                if header:                          # If we already found the header
                                                        # If the length of the space is extra long
                    if chk_tol(sequence[1], rules['quiet'], 10000):
                        bursts.extend(burst)          # Assume we've finished processing a burst
                        burst = []                    # Init to prepare for the next burst
                        header = False                # A new burst means a new header, so set to false
                        continue                      # We got a burst! lets look for the next one
                    
                    # Otherwise, keep appending bits
                    bits.append(str(chk_tol(sequence[1], rules['space0'], rules['delta'])))
                    
                    # Each burst will contain a max of 16 bits, so if we're at 16
                    if len(bits) == rules['bits']: 
                        
                        # Then it's time to store this burst and prepare for the next one
                        burst.append(hex(int(''.join(bits), 2)).split('x')[1].upper())  # Output to HEX
#                         burst.append(int(''.join(bits), 2))                           # Output to DEC
#                         burst.append(''.join(bits))                                   # Output to BIN
                        bits = []                     # New burst means new bits - init
                        continue
                
            elif sequence[2]:                    # If a mark... We don't really care about marks...
                if not header:                     # Unless we haven't found the header
                    
                                                     # We totally care about that mark
                    if chk_tol(sequence[1], rules['header'], rules['delta']):
                        
                        header = True                  # Time to collect some bits...
                        continue
                        
        samples.append(bursts)
    decoded.update({command: samples})
dfs = {key: pd.DataFrame(decoded[key]) for key in decoded.keys()}
{print(f'{key}:\n {dfs[key]}\n\n') for key in decoded.keys()}

send_dall_30x:
     0    1    2    3    4    5    6    7    8    9    10   11
0  513  114  615  316  217  418  519  11A  61B  31C  21D  41E
1  513  114  615  316  217  418  519  11A  61B  31C  21D  41E
2  513  114  615  316  217  418  519  11A  61B  31C  21D  41E


send_F_B_F_B_confirm_pair:
       0     1     2    3    4    5    6    7    8    9    10    11    12  \
0   3F00  3F00  3F00  2B1  2B2  8B3  2B4  8B5  501  102  603   304   205   
1   3F00  3F00  3F00  2B1  2B2  8B3  2B4  8B5  501  102  603   304   205   
2   3F00  3F00  3F00  2B1  2B2  8B3  2B4  8B5  501  102  603   304   205   
3   3F00  3F00  3F00  2B1  2B2  8B3  2B4  8B5  501  102  603   304   205   
4   3F00  3F00  3F00  2B1  2B2  8B3  2B4  8B5  501  102  603   304   205   
5   3F00  3F00  3F00  2B1  2B2  8B3  2B4  8B5  501  102  603   304   205   
6   3F00  3F00  3F00  2B1  2B2  8B3  2B4  8B5  501  102  603   304   205   
7   3F00  3F00  3F00  2B1  2B2  8B3  2B4  8B5  501  102  603   304   205   
8   3F00  3F00  3F00  

{None}

In [437]:
def decode_signal(commands):
# Decode the contents of each transmission sample
    decd = {}
    for command, history in commands.items():
        samples = []
        for series in history:
            bursts = []
            burst = []
            header = False
            bits = []

            for sequence in series:
                if not sequence[2]:                   # If a space
                    if header:                          # If we already found the header
                                                        # If the length of the space is extra long
                        if chk_tol(sequence[1], rules['quiet'], 10000):
                            bursts.extend(burst)          # Assume we've finished processing a burst
                            burst = []                    # Init to prepare for the next burst
                            header = False                # A new burst means a new header, so set to false
                            continue                      # We got a burst! lets look for the next one

                        # Otherwise, keep appending bits
                        bits.append(str(chk_tol(sequence[1], rules['space0'], rules['delta'])))

                        # Each burst will contain a max of 16 bits, so if we're at 16
                        if len(bits) == rules['bits']: 
#                             print('Found a packet!')
                            # Then it's time to store this burst and prepare for the next one
                            burst.append(hex(int(''.join(bits), 2)).split('x')[1].upper())  # Output to HEX
    #                         burst.append(int(''.join(bits), 2))                           # Output to DEC
    #                         burst.append(''.join(bits))                                   # Output to BIN
                            bits = []                     # New burst means new bits - init
                            continue

                elif sequence[2]:                    # If a mark... We don't really care about marks...
                    if not header:                     # Unless we haven't found the header

                                                         # We totally care about that mark
                        if chk_tol(sequence[1], rules['header'], rules['delta']):
#                             print('Found the header!')
                            header = True                  # Time to collect some bits...
                            continue

            samples.append(bursts)
        decd.update({command: samples})
    dfs = {key: pd.DataFrame(decd[key]) for key in decd.keys()}
    {print(f'{key}:\n {dfs[key]}\n\n') for key in decd.keys()}
    return decd

In [439]:
signals = {"light": [4170, 662, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 24863, 4170, 662, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 24863, 4170, 662, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 24863, 4170, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 1281, 656, 662, 656, 24863, 4170, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 1281, 656, 24863, 4170, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 24863, 4170, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 1281, 656, 24863, 4170, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 24863, 4170, 662, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 662, 656, 24863, 4170, 662, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 662, 656, 24863],
 "light2": [4170, 662, 656, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 24863, 4170, 662, 656, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 24863, 4170, 662, 656, 662, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 24863, 4170, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 24863, 4170, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 24863, 4170, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 24863, 4170, 662, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 662, 656, 24863, 4170, 662, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 1281, 656, 662, 656, 1281, 656, 18170, 4170, 662, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 18170, 4170, 662, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 1281, 656, 18170, 4170, 662, 656, 662, 656, 662, 656, 1281, 656, 1281, 656, 662, 656, 662, 656, 1281, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 662, 656, 1281, 656],
 "light3": [4170, 662, 603, 662, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 24863, 4170, 662, 603, 662, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 24863, 4170, 662, 603, 662, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 24863, 4170, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 24863, 4170, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 24863, 4170, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 1281, 603, 1281, 603, 24863, 4170, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 24863, 4170, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 18170, 4170, 662, 603, 662, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 18170, 4170, 662, 603, 662, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 18170, 4170, 662, 603, 662, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603],
 "light4": [4170, 662, 603, 662, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 24863, 4170, 662, 603, 662, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 24863, 4170, 662, 603, 662, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 24863, 4170, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 24863, 4170, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 24863, 4170, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 1281, 603, 1281, 603, 24863, 4170, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 24863, 4170, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 1281, 603, 662, 603, 1281, 603, 18170, 4170, 662, 603, 662, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 18170, 4170, 662, 603, 662, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603, 18170, 4170, 662, 603, 662, 603, 662, 603, 1281, 603, 1281, 603, 662, 603, 662, 603, 1281, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 662, 603, 1281, 603],
    "bf2": [4142, 683, 609, 683, 609, 1299, 609, 1299, 609, 1299, 609, 1299, 609, 1299, 609, 1299, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 24748, 4142, 683, 609, 683, 609, 1299, 609, 1299, 609, 1299, 609, 1299, 609, 1299, 609, 1299, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 24748, 4142, 683, 609, 683, 609, 1299, 609, 1299, 609, 1299, 609, 1299, 609, 1299, 609, 1299, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 24748, 4142, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 683, 609, 1299, 609, 683, 609, 1299, 609, 1299, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 24748, 4142, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 683, 609, 1299, 609, 683, 609, 1299, 609, 1299, 609, 683, 609, 683, 609, 1299, 609, 683, 609, 24748, 4142, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 683, 609, 1299, 609, 683, 609, 1299, 609, 1299, 609, 683, 609, 683, 609, 1299, 609, 1299, 609, 24748, 4142, 683, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 683, 609, 1299, 609, 1299, 609, 683, 609, 1299, 609, 683, 609, 683, 609, 24748, 4142, 683, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 683, 609, 1299, 609, 1299, 609, 683, 609, 1299, 609, 683, 609, 1299, 609, 24748, 4142, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 24748, 4142, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 683, 609, 18093, 4142, 683, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 1299, 609, 1299, 609, 1299, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 683, 609, 1299, 609, 1299, 609]}


In [407]:
bits

[array([   0, 4175,    1]),
 array([  1, 652,   0]),
 array([  2, 656,   1]),
 array([  3, 652,   0]),
 array([  4, 656,   1]),
 array([   5, 1275,    0]),
 array([  6, 656,   1]),
 array([   7, 1275,    0]),
 array([  8, 656,   1]),
 array([   9, 1275,    0]),
 array([ 10, 656,   1]),
 array([  11, 1275,    0]),
 array([ 12, 656,   1]),
 array([  13, 1275,    0]),
 array([ 14, 656,   1]),
 array([  15, 1275,    0]),
 array([ 16, 656,   1]),
 array([ 17, 652,   0]),
 array([ 18, 656,   1]),
 array([ 19, 652,   0]),
 array([ 20, 656,   1]),
 array([ 21, 652,   0]),
 array([ 22, 656,   1]),
 array([ 23, 652,   0]),
 array([ 24, 656,   1]),
 array([ 25, 652,   0]),
 array([ 26, 656,   1]),
 array([ 27, 652,   0]),
 array([ 28, 656,   1]),
 array([ 29, 652,   0]),
 array([ 30, 656,   1]),
 array([ 31, 652,   0]),
 array([ 32, 656,   1]),
 array([   33, 24882,     0]),
 array([  34, 4175,    1]),
 array([ 35, 652,   0]),
 array([ 36, 656,   1]),
 array([ 37, 652,   0]),
 array([ 38, 656,   

In [440]:
# key_name = 'light2'
from_pi = {} 

for key_name, data in signals.items():
    value = True
    bits = []
    for i, pulse in enumerate(data):
        bits.append(np.array([i, pulse, int(value)]))
        value = not value
    
    from_pi[key_name] = [bits]

decode_signal(from_pi)

light:
       0     1     2     3     4     5     6     7     8     9
0  40FF  40FF  40FF  7D4E  7D4D  7D4C  7D4B  7D4A  66FE  66FE


light2:
       0     1     2    3    4    5    6    7     8     9
0  3F00  3F00  3F00  2B1  2B2  2B3  8B4  8B5  1901  1901


light3:
       0     1     2    3    4    5    6    7     8     9
0  3F00  3F00  3F00  2B1  2B2  2B3  8B4  8B5  1901  1901


light4:
       0     1     2    3    4    5    6    7     8     9
0  3F00  3F00  3F00  2B1  2B2  2B3  8B4  8B5  1901  1901


bf2:
       0     1     2    3    4    5    6    7    8    9
0  3F00  3F00  3F00  2B1  2B2  2B3  8B4  8B5  201  102




{'light': [['40FF',
   '40FF',
   '40FF',
   '7D4E',
   '7D4D',
   '7D4C',
   '7D4B',
   '7D4A',
   '66FE',
   '66FE']],
 'light2': [['3F00',
   '3F00',
   '3F00',
   '2B1',
   '2B2',
   '2B3',
   '8B4',
   '8B5',
   '1901',
   '1901']],
 'light3': [['3F00',
   '3F00',
   '3F00',
   '2B1',
   '2B2',
   '2B3',
   '8B4',
   '8B5',
   '1901',
   '1901']],
 'light4': [['3F00',
   '3F00',
   '3F00',
   '2B1',
   '2B2',
   '2B3',
   '8B4',
   '8B5',
   '1901',
   '1901']],
 'bf2': [['3F00',
   '3F00',
   '3F00',
   '2B1',
   '2B2',
   '2B3',
   '8B4',
   '8B5',
   '201',
   '102']]}

In [17]:
def check_bit(length):
    if length > 6000:
        return 'Silence'
    
    if 5000 > length > 3500:
        return 'Header'
    
    if 1500 > length > 1050:
        return 1
    
    if 850 > length > 450:
        return 0
    
    return 'Error'

In [399]:
value = True
bits = []
for i, pulse in enumerate(signals['light2']):
    bits.append(np.array([i, pulse, int(value)]))
    value = not value
bots = bits

In [392]:
['0x3f00', '0x3f00', '0x3f00', '0x2b1', '0x2b2', '0x2b3', '0x2b4', '0x2b5', '0x8201', '0x8102']

[['M' 'S' 'M' ... 'M' 'S' 'M']
 [4175 652 656 ... 656 1275 656]
 ['Header' 0 None ... None 1 None]]


In [393]:
for event in bits:
    _type = event[0]
    length = event[1]
    
    res = check_bit(length)
    if isinstance(res, str):
        event.append(res)
        continue
    else:
        if _type == 'S':
            # If we have a space and it is long, append 1. Otherwise append 0
            event.append(1) if res else event.append(0)
    
        if _type == 'M':
            # -1 will indicate a decode error since marks should never be this long
            event.append(-1) if res else event.append(None)
bits

wave = []
labels = []
# fig, ax = plt.subplot()
for e in bits:
    val = 1 if e[0] == 'M' else 0
    wave.extend([val] * e[1])
# ax.line([for e in bits])
plt.plot(wave)


In [430]:
_ = [[print(f'{k}:\n {[print(decoded[k][i]) for i in range(len(decoded[k]))]}\n') for key in ['send',] if key in k ] for k in decoded.keys()]

['3F00', '3F00', '3F00', '2B1', '2B2', '2B3', '8B4', '8B5', '501', '102', '603', '304', '205', '406']
send_L4_F_R4_L9_B_R9_after_pair:
 [None]

['BF00', 'BF00', 'BF00', '82B1', '82B2', '82B3', '82B4', '82B5', '8501', '8102', '8603', '8304', '8205', '8406', '8507', '8108', '8609', '830A', '820B', '840C']
send_L4_F_R4_L9_B_R9_OD_ON_L4_F_R4_L9_B_R9_OD-OFF_L4_F_R4_L9_B_R9:
 [None]

['BF00', 'BF00', 'BF00', '82B1', '82B2', '82B3', '82B4', '82B5', '8501', '8102', '8603', '8304', '8205', '8406']
['BF00', 'BF00', 'BF00', '82B1', '82B2', '82B3', '82B4', '82B5', '8501', '8102', '8603', '8304', '8205', '8406']
['BF00', 'BF00', 'BF00', '82B1', '82B2', '82B3', '82B4', '82B5', '8501', '8102', '8603', '8304', '8205', '8406']
send_L4_F_R4_L9_B_R9:
 [None, None, None]

['BF00', 'BF00', 'BF00', '82B1', '82B2', '82B3', '82B4', '82B5', '8101', '8102', '8103', '8104']
send_FFFF:
 [None]

['BF00', 'BF00', 'BF00', '82B1', '82B2', '82B3', '82B4', '82B5', '8101']
send_F:
 [None]

['9701', '9701']
send_empty:
 

In [79]:
decoded

{'send_F_B_F_B_confirm_pair': [['3F00',
   '3F00',
   '3F00',
   '2B1',
   '2B2',
   '2B3',
   '8B4',
   '8B5',
   '101',
   '202',
   '103',
   '204'],
  ['3F00',
   '3F00',
   '31B1',
   '2B2',
   '8B3',
   '2B4',
   '8B5',
   '2B6',
   '8B7',
   '2B8',
   '8B9'],
  ['3F00',
   '3F00',
   '31B1',
   '2B2',
   '2B3',
   '8B4',
   '8B5',
   '2B6',
   '2B7',
   '8B8',
   '8B9'],
  ['3F00',
   '3F00',
   '3F00',
   '2B1',
   '2B2',
   '2B3',
   '8B4',
   '8B5',
   '101',
   '202',
   '103',
   '204'],
  ['3F00',
   '3F00',
   '3F00',
   '2B1',
   '2B2',
   '2B3',
   '8B4',
   '8B5',
   '101',
   '202',
   '103',
   '204']],
 'send_L4_F_R4_L9_B_R9_after_pair': [['3F00',
   '3F00',
   '3F00',
   '2B1',
   '2B2',
   '2B3',
   '8B4',
   '8B5',
   '501',
   '102',
   '603',
   '304',
   '205',
   '406']],
 'clear_after_pair': [['3F00',
   '3F00',
   '3F00',
   '2B1',
   '2B2',
   '2B3',
   '8B4',
   '8B5',
   '1601',
   '1601']],
 'pairing_init': [['0', '0', '0', '3B1', 'AB2', 'DB3', 'AB4', '

send_F_B_F_B_confirm_pair:
      0     1     2    3    4    5    6    7    8    9    10    11
0  3F00  3F00  3F00  2B1  2B2  2B3  8B4  8B5  101  202  103   204
1  3F00  3F00  31B1  2B2  8B3  2B4  8B5  2B6  8B7  2B8  8B9  None
2  3F00  3F00  31B1  2B2  2B3  8B4  8B5  2B6  2B7  8B8  8B9  None
3  3F00  3F00  3F00  2B1  2B2  2B3  8B4  8B5  101  202  103   204
4  3F00  3F00  3F00  2B1  2B2  2B3  8B4  8B5  101  202  103   204


send_L4_F_R4_L9_B_R9_after_pair:
      0     1     2    3    4    5    6    7    8    9    10   11   12   13
0  3F00  3F00  3F00  2B1  2B2  2B3  8B4  8B5  501  102  603  304  205  406


clear_after_pair:
       0     1     2    3    4    5    6    7     8     9
0  3F00  3F00  3F00  2B1  2B2  2B3  8B4  8B5  1601  1601


pairing_init:
    0  1  2    3    4    5    6    7    8
0  0  0  0  3B1  AB2  DB3  AB4  DB5  101
1  0  0  0  3B1  AB2  DB3  AB4  DB5  101


pairing_timeout:
       0     1     2    3    4    5    6    7
0  3F00  3F00  3F00  2B1  2B2  2B3  2B4  2B5


Cle

{None}

In [274]:
val = f'{int("0xBF00", 16):>b}'
cut = int(len(val)/2)
print(f'{len(val)}: {val}')
print(cut)
print(str(val[:cut]) + ' ' + str(val[cut:]))

16: 1011111100000000
8
10111111 00000000


In [269]:
bin(int('0xbf00', 16))

'0b1011111100000000'

In [125]:
def build_address(addr):
    rtrn_qty = 3
    bits = [addr, 'F', '0', '0']
    return [hex(int(''.join(bits), 16))] * rtrn_qty

In [137]:
def build_preamble(addr):
    rtrn_qty = 5
    byts = [[str(int(addr, 16) - 3), '2', 'B', str(i) ] for i in range(1, rtrn_qty + 1)]
    return [hex(int(''.join(byte), 16)) for byte in byts]

In [134]:
(int('f', 16))

15

In [427]:

ch_map = {0:'3', 1:'7', 2:'B', 3:'F'}

ch = 0

command = []
command.extend(build_addr(ch_map[ch]))
command.extend(build_pre(ch_map[ch]))
# command.extend(build_cmd())
print(command)

['0x3f00', '0x3f00', '0x3f00', '0x2b1', '0x2b2', '0x2b3', '0x2b4', '0x2b5']


In [155]:
def parse_direction(direction: str):
    # Accepts a direction command and returns the corresponding binary value that is transmitted to Botley.
    direction = direction.lower()  # To minimize errors, make sure direction is always lower case

    # The "move" dictionary maps simplified commands to the binary values that will be sent to Botley
    move = {'f': ('1', 'forward'), 'b': ('2', 'backward'),
            'l90': ('3', 'left90'), 'r90': ('4', 'right90'),
            'l45': ('5', 'left45'), 'r45': ('6', 'right45')}

    # The "commands" dictionary maps similar direction inputs to their common simplified "move" commands.
    # This allows us to say botley.move('forward') or botley.move('f'); both will move Botley forward.
    commands = {'forward': move['f'], 'backward': move['b'], 'f': move['f'], 'b': move['b'],
                'left90': move['l90'], 'right90': move['r90'], 'l9': move['l90'], 'r9': move['r90'],
                'left45': move['l45'], 'right45': move['r45'], 'l4': move['l45'], 'r4': move['r45'],
                'l': move['l90'], 'r': move['r90']}

    # Next, "commands[direction]" looks for a "key" in "commands" that matches the "direction" parameter given.
    # If the "key" is found, the corresponding "value" is returned. In this case, the return value is a "key"
    # in the "move" dictionary. Because "commands[direction]" is inside of "move[]", the return value of
    # "commands[direction]" is passed as input to "move[]". "move[]" returns a list containing the
    # corresponding binary move command and a string that we can use to label the move when we print the queue.
    command = commands[direction]

    return command

In [189]:
def build_message(channel, commands):
    # Construct the header which is sent with each transmission
    message_header = build_message_header(channel)
    
    # Convert the user commands into hex
    user_commands = build_commands(commands.replace(' ','').split(','))
    
    # Combine the header and the commands to create the entire message
    hex_message = message_header + user_commands
    
    return hex_message
    

In [252]:
def build_message_header(channel):
    address_bytes = build_addr(channel)
    preamble_bytes = build_pre(channel)
    return address_bytes + preamble_bytes

In [253]:
def build_commands(commands):
    hex_commands = []
    for i, command in enumerate(commands):
        cmd_bit = parse_direction(command)[0]
        byte = ['8', cmd_bit, f'{i+1:02d}']
        hex_commands.append(hex(int(''.join(byte), 16)))
    return hex_commands

In [368]:
def encode_botley(message):
    """ Encodes the contents of the message """
    encoded = []
    for data in message:
        binary = f'{int(data, 16):16b}'.replace(' ', '0')
        encode = [[rules['space1'], rules['mark']] if int(bit) else [rules['space0'], rules['mark']] for bit in binary]
        encode = np.array(encode).ravel().tolist()
        encode = [rules['header']] + encode + [rules['quiet']]
        encoded.extend(encode)   
    return encoded

In [441]:
msg = build_message(ch_map[ch], 'b, f')
print(f'{len(msg)}: {msg}')

10: ['0x3f00', '0x3f00', '0x3f00', '0x2b1', '0x2b2', '0x2b3', '0x2b4', '0x2b5', '0x8201', '0x8102']


In [430]:
[f'{len(b)}: {b}' for b in encode_botley(msg)]

TypeError: object of type 'int' has no len()

In [429]:
np.array([[rules['space1'], rules['mark']] if int(bit) else [rules['space0'], rules['mark']] for bit in f'{int(msg[0], 16):b}']).ravel().tolist()

[642,
 642,
 642,
 642,
 642,
 642,
 642,
 642,
 642,
 642,
 642,
 642,
 1284,
 642,
 1284,
 642,
 1284,
 642,
 1284,
 642,
 1284,
 642,
 1284,
 642,
 1284,
 642,
 1284,
 642]

In [340]:
n = [[1, rules['mark']] if int(bit) else [0, rules['mark']] for bit in f'{int(msg[0], 16):b}']
print(f'{len(n)}: {n}')

15: [[1, 642], [1, 642], [1, 642], [1, 642], [1, 642], [1, 642], [1, 642], [0, 642], [0, 642], [0, 642], [0, 642], [0, 642], [0, 642], [0, 642], [0, 642]]


In [377]:
decoded['light'][0]

['BF00',
 'BF00',
 'BF00',
 '82B1',
 '82B2',
 '82B3',
 '82B4',
 '82B5',
 '9901',
 '9901']

In [431]:
encoded = [val for val in encode_botley(msg)]
print(encoded)

[4150, 1284, 642, 1284, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 5000, 4150, 1284, 642, 1284, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 5000, 4150, 1284, 642, 1284, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 5000, 4150, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 642, 642, 1284, 642, 642, 642, 1284, 642, 642, 642, 642, 642, 1284, 642, 1284, 642, 1284, 642, 642, 642, 5000, 4150, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 642, 642, 1284, 642, 642, 642, 1284, 642, 642, 642, 642, 642, 1284, 642, 1284, 642, 642, 642, 1284, 642, 5000, 4150, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 642, 642, 1284,

In [432]:
print(json.dumps({"f_b": encoded}))
                  

{"f_b": [4150, 1284, 642, 1284, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 5000, 4150, 1284, 642, 1284, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 5000, 4150, 1284, 642, 1284, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 5000, 4150, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 642, 642, 1284, 642, 642, 642, 1284, 642, 642, 642, 642, 642, 1284, 642, 1284, 642, 1284, 642, 642, 642, 5000, 4150, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 642, 642, 1284, 642, 642, 642, 1284, 642, 642, 642, 642, 642, 1284, 642, 1284, 642, 642, 642, 1284, 642, 5000, 4150, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 1284, 642, 642, 64

In [442]:
address_map = {
    # The pairing sequence uses the F, L, R, and B direction buttons to create an address.
    # Each direction button corresponds to a HEX value which is used as part of the address.
    # There are 256 possible permutations of button presses which will result in a unique address.
    # This dict will make it easy to convert between direction/HEX addresses
    'F': '2',       # Forward
    'L': '4',       # Left
    'R': '6',       # Right
    'B': '8'        # Backward
}

In [459]:
address = 'blfr'
allowed = 'FLRB2468'
converted = []
address = address.upper()
if len(address) != 4:
    print(f'You provided {address} which is {len(address)} characters. The address must be 4 characters long.')
    return None

if all(character in allowed for character in address):
    if address.isnumeric():
        for _hex in address:
            [converted.append(key) for key, value in address_map.items() if value == _hex]

    else:
        converted = [address_map[character] for character in address]
converted

['8', '4', '2', '6']